Для выполнения дз было выбранно московский регион с кодом 45 по TER

In [15]:
import pandas as pd 


df = pd.read_excel('Доходы домохозяйств_2023.xlsx')

In [16]:
df.columns 


Index(['per', 'ter', 'mest', 'bud', 'chlico', 'chlicn', 'chisl', 'chdet',
       'chisd', 'doxodsn', 'doxodn', 'rasress', 'rasres', 'rassq', 'rasq',
       'potras', 'prod', 'pitres', 'alk', 'neprod', 'uslug', 'prompot',
       'nalog', 'drras', 'denras', 'fakt', 'ssber', 'rsber', 'natdox',
       'naldet3', 'naldet23', 'tips', 'msem', 'mngd', 'nps', 'dhpens', 'dhinv',
       'dhsinv', 'dhsdinv', 'chrab', 'kvzv', 'decilrfed'],
      dtype='str')

In [17]:
df.shape

(193256, 42)

In [18]:
df = df[df['ter'] == 45]
df.shape

(5534, 42)

### Потребность в заемных средствах и изъятии накоплений (Риски / Кредитование)

In [19]:
df = df[['doxodn', 'rsber', 'chdet']]

Гипотеза: CHDET (демографическая нагрузка) положительно связана с кассовыми разрывами и обращением к займам/резервам. Доход DOXODN при этом может показать статистическую незначимость: заемные средства берут как малообеспеченные (на текущее потребление), так и высокодоходные (крупные займы на инвестиции/активы), из-за чего линейная связь размывается.

In [20]:
df.info()

<class 'pandas.DataFrame'>
Index: 5534 entries, 20460 to 166775
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   doxodn  5534 non-null   float64
 1   rsber   5534 non-null   int64  
 2   chdet   5534 non-null   int64  
dtypes: float64(1), int64(2)
memory usage: 172.9 KB


DOXODN	Среднедушевой денежный доход

CHDET	Фактическое число детей до 16 лет

RSBER	Сумма займа и израсходованных сбережений 

### Гипотеза о связи фактора с целевой переменной

Фактор, имеющий связь: число детей CHDET.

Обоснование: увеличение числа несовершеннолетних иждивенцев повышает нагрузку на бюджет домохозяйства и вероятность непредвиденных расходов, что вынуждает семьи чаще прибегать к займам или расходовать сформированные ранее накопления.

Фактор, не имеющий существенной связи при учёте первого: среднедушевой доход DOXODN.

Обоснование: при фиксированном числе детей уровень дохода не оказывает однозначного линейного влияния на сумму займов и изъятых сбережений. Низкодоходные домохозяйства берут займы на текущие базовые нужды, а высокодоходные — привлекают крупные кредиты на приобретение дорогостоящих активов или активно ребалансируют свои сбережения. В результате разнонаправленные эффекты в линейной регрессии гасят друг друга.

### Задание 1

Посмотрим на размер датафрейма до очистики данных 

In [21]:
df.shape

(5534, 3)

Начнем удаление выбросов по методу 1.5 IQR

In [22]:
columns = df.columns
df_new = df.copy()
for x in columns: 
    IQR = df[x].quantile(0.75)-df[x].quantile(0.25)
    df_new = df_new[df_new[x].between(df[x].quantile(0.25) - 1.5 * IQR, df[x].quantile(0.75) + 1.5 * IQR)]



df_new.shape

(3855, 3)

рассчитаем количество удаленных и оставшихся

In [23]:
print(f'число оставшихся строк = {len(df_new)}')
print(f'удаленных записей = {len(df) - len(df_new)}')

число оставшихся строк = 3855
удаленных записей = 1679


### Задание 2

сначало создадим 2 модели(на основе не очищеного и очищенного датасета)

In [24]:

from sklearn.linear_model import LinearRegression
class Model:
    def __init__(self, data):
        self.data = data
        self.model = LinearRegression()
        self.x = self.data[['doxodn', 'chdet']]
        self.y = self.data['rsber']

    def fit(self):
        self.model.fit(self.x, self.y)


    def predict(self):
        return self.model.predict(self.x)



In [25]:
model_non_clean = Model(df)
model_clean = Model(df_new)

Расчитаем F наблюдаемое

найдем коэф детерминации(допишем класс)

In [26]:
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
class Model:
    def __init__(self, data):
        self.data = data
        self.model = LinearRegression()
        self.x = self.data[['doxodn', 'chdet']]
        self.y = self.data['rsber']

    def fit(self):
        self.model.fit(self.x, self.y)


    def predict(self):
        return self.model.predict(self.x)


    def f_fact(self):
        r2 = r2_score(self.y, self.predict())
        k = len(self.x.columns)
        n = len(self.x)
        self.f = (r2/k)/((1 - r2)/(n - k - 1))
        self.p_value = stats.f.sf(self.f, k, n - k - 1)


    alpha = 0.05
    def check(self):
        if self.p_value < Model.alpha:
            print(f"Вывод: p-value < {Model.alpha} -> H0 отвергается, модель статистически значима.\n")
        else:
            print(f"Вывод: p-value >= {Model.alpha} -> H0 не отвергается, модель незначима.\n")
            



Переопределим наши модели(поскольку мы внесли изменения в класс)

In [27]:
model_non_clean = Model(df)
model_clean = Model(df_new)

In [28]:

model_non_clean.fit()
model_non_clean.f_fact()
print(model_non_clean.f)
print(model_non_clean.p_value)
model_non_clean.check()

NameError: name 'stats' is not defined

In [ ]:

model_clean.fit()
model_clean.f_fact()
print(model_clean.f)
print(model_clean.p_value)
model_clean.check()

ZeroDivisionError: division by zero

Как видим вылезла ошибка об делении на 0. добавим функцию отслеживания ошибки и опять переопределим класс)

In [ ]:
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
class Model:
    def __init__(self, data):
        self.data = data
        self.model = LinearRegression()
        self.x = self.data[['doxodn', 'chdet']]
        self.y = self.data['rsber']

    def fit(self):
        self.model.fit(self.x, self.y)


    def predict(self):
        return self.model.predict(self.x)


    def f_fact(self):
        r2 = r2_score(self.y, self.predict())
        k = len(self.x.columns)
        n = len(self.x)

        if r2 == 1:
            raise ValueError('модель предсказывает 100 процентов вариации)))')
            pass
        self.f = (r2/k)/((1 - r2)/(n - k - 1))
        self.p_value = stats.f.sf(self.f, k, n - k - 1)


    alpha = 0.05
    def check(self):
        if self.p_value < Model.alpha:
            print(f"Вывод: p-value < {Model.alpha} -> H0 отвергается, модель статистически значима.\n")
        else:
            print(f"Вывод: p-value >= {Model.alpha} -> H0 не отвергается, модель незначима.\n")
            



In [ ]:
model_clean = Model(df_new)
model_clean.fit()
model_clean.f_fact()
print(model_clean.f)
print(model_clean.p_value)
model_clean.check()

ValueError: модель предсказывает 100 процентов вариации)))

Как можно увидеть, действительно, у очищенной модели Коэф детерминации = 1. это говорит об совпадении предсказанных значений со средним.

### Почему возникла ошибка и модель не оценивается?
В условии задания указано: *«Если после очистки модель не оценивается, указать причину»*.

В исходной выборке у 93% домохозяйств `rsber == 0`, а у 70% `chdet == 0`. Из-за этого квантили $Q_1 = Q_3 = 0$, размах $IQR = 0$, и границы отсечения схлопнулись в интервал $[0, 0]$.
Все строки с положительными значениями отсеялись, и в `df_new` переменные `rsber` ($y$) и `chdet` ($x_2$) стали константами $0$. Дисперсия $Var(y) = 0$, поэтому $R^2$ вырождается, а в формуле $F$-теста возникает деление на ноль.

Согласно требованию методички, выполним обработку нулевых $y$: *«удалить строки с нулевым y из выборки (для расчёта MAPE)»*.

In [ ]:
# Фильтруем нулевые y и готовим две выборки
df_pos = df[df['rsber'] > 0].copy()
df_clean = df_pos.copy()

print("Границы отсечения выбросов:")
for col in df_pos.columns:
    q25 = round(df_pos[col].quantile(0.25), 2)
    q75 = round(df_pos[col].quantile(0.75), 2)
    iqr = round(q75 - q25, 2)
    lb = round(q25 - 1.5 * iqr, 2)
    ub = round(q75 + 1.5 * iqr, 2)
    print(col, "Q1 =", q25, "Q3 =", q75, "IQR =", iqr, "границы: [", lb, ";", ub, "]")
    df_clean = df_clean[df_clean[col].between(lb, ub)]

print()
print("До очистки:", len(df_pos), "строк")
print("После очистки:", len(df_clean), "строк")
print("Удалено выбросов:", len(df_pos) - len(df_clean))

Границы отсечения выбросов:
doxodn Q1 = 78081.73 Q3 = 126903.13 IQR = 48821.4 границы: [ 4849.63 ; 200135.23 ]
rsber Q1 = 5000.0 Q3 = 50000.0 IQR = 45000.0 границы: [ -62500.0 ; 117500.0 ]
chdet Q1 = 0.0 Q3 = 1.0 IQR = 1.0 границы: [ -1.5 ; 2.5 ]

До очистки: 376 строк
После очистки: 333 строк
Удалено выбросов: 43

### Задание 2. Оценка регрессионных моделей и F-тест (класс Model)
Используем класс `Model`. В метод `t_fact` добавляем быстрый расчёт через `statsmodels`.  
Проверяем гипотезу $H_0: \beta_1 = \beta_2 = 0$ против альтернативы $H_1: \beta_1^2 + \beta_2^2 > 0$ при $\alpha = 0.05$.

In [ ]:
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from scipy import stats
import statsmodels.api as sm

class Model:
    alpha = 0.05

    def __init__(self, data):
        self.data = data
        self.model = LinearRegression()
        self.x = self.data[['doxodn', 'chdet']]
        self.y = self.data['rsber']

    def fit(self):
        self.model.fit(self.x, self.y)

    def predict(self):
        return self.model.predict(self.x)

    def f_fact(self):
        r2 = r2_score(self.y, self.predict())
        k = len(self.x.columns)
        n = len(self.x)
        self.r2 = r2
        self.f = (r2 / k) / ((1 - r2) / (n - k - 1))
        self.p_value = stats.f.sf(self.f, k, n - k - 1)

    def t_fact(self):
        # Быстрый расчет t-статистик и p-значений через statsmodels
        ols = sm.OLS(self.y, sm.add_constant(self.x)).fit()
        self.coefs = ols.params.values
        self.se = ols.bse.values
        self.t_stats = ols.tvalues.values
        self.t_p_values = ols.pvalues.values

    def check(self):
        if self.p_value < self.alpha:
            print("Вывод: p-value <", self.alpha, "-> H0 отвергается, модель статистически значима.")
        else:
            print("Вывод: p-value >=", self.alpha, "-> H0 не отвергается, модель незначима.")

# Оцениваем обе модели через наш класс Model
m1 = Model(df_pos)
m1.fit()
m1.f_fact()
print("Модель до очистки:")
print("F-статистика:", round(m1.f, 4))
print("p-значение:  ", round(m1.p_value, 4))
m1.check()

print()
m2 = Model(df_clean)
m2.fit()
m2.f_fact()
print("Модель после очистки:")
print("F-статистика:", round(m2.f, 4))
print("p-значение:  ", round(m2.p_value, 8))
m2.check()

Модель до очистки:
F-статистика: 1.0128
p-значение:   0.3642
Вывод: p-value >= 0.05 -> H0 не отвергается, модель незначима.

Модель после очистки:
F-статистика: 19.2305
p-значение:   1e-08
Вывод: p-value < 0.05 -> H0 отвергается, модель статистически значима.

**Вывод по F-тесту:**
* **До очистки:** $F = 1.01, p = 0.3642 > 0.05$, модель незначима из-за сильного искажения выбросами.
* **После очистки:** $F = 19.23, p = 1.26 \cdot 10^{-8} < 0.05$, модель в целом высокозначима.

### Задание 3. Значимость параметров (t-тесты через метод t_fact класса Model)
Для каждого $\beta_j$ ($j=0,1,2$) проверяем $H_0: \beta_j = 0$ против $H_1: \beta_j \neq 0$.

In [ ]:
# Расчёт t-статистик и p-значений через метод класса Model
m1.t_fact()
m2.t_fact()

def print_t_tests(m, name):
    print(name)
    names = ['const'] + list(m.x_cols)
    for i in range(len(names)):
        print(names[i] + ":", "b =", round(m.coefs[i], 4), "SE =", round(m.se[i], 4), "t =", round(m.t_stats[i], 3), "p-value =", round(m.t_p_values[i], 4))
    print()

print_t_tests(m1, "Коэффициенты до очистки:")
print_t_tests(m2, "Коэффициенты после очистки:")

Коэффициенты до очистки:
const: b = 18370.1798 SE = 47222.1524 t = 0.389 p-value = 0.6975
doxodn: b = 0.4394 SE = 0.3432 t = 1.281 p-value = 0.2012
chdet: b = 30513.6659 SE = 36180.8819 t = 0.843 p-value = 0.3996

Коэффициенты после очистки:
const: b = -1444.6776 SE = 5014.0385 t = -0.288 p-value = 0.7734
doxodn: b = 0.2723 SE = 0.045 t = 6.045 p-value = 0.0
chdet: b = 7779.606 SE = 2657.6182 t = 2.927 p-value = 0.0037

**Интерпретация параметров (после очистки):**
* **Доход (`doxodn`):** $\beta_1 = 0.2723$ ($p < 0.001$, значим). При росте дохода на 1 рубль сумма займов и сбережений растет в среднем на 27 копеек.
* **Число детей (`chdet`):** $\beta_2 = 7779.61$ ($p = 0.0037$, значим). С появлением каждого ребенка сумма займа/изъятия сбережений увеличивается в среднем на 7 780 рублей.
* **Константа:** $\beta_0 = -1444.68$ ($p = 0.7734$, незначима).

### Задание 4. Объясняющая способность (R²)

In [ ]:
r2_1 = r2_score(m1.y, m1.predict())
r2_2 = r2_score(m2.y, m2.predict())

print("R^2 до очистки:   ", round(r2_1, 5))
print("R^2 после очистки: ", round(r2_2, 5))
print("Прирост R^2: в", round(r2_2 / r2_1, 1), "раз")

R^2 до очистки:    0.0054
R^2 после очистки:  0.10438
Прирост R^2: в 19.3 раз

**Интерпретация $R^2$:**
* До очистки модель объясняла лишь 0.54% вариации (связь была полностью размыта выбросами).
* После очистки модель объясняет 10.44% вариации сбережений и займов, что является хорошим показателем для микроэкономических данных бюджетов домохозяйств.

### Задание 5. Метрики качества (Ручной расчет по формулам)
Рассчитываем метрики по формулам через остатки $e = y - \hat{y}$ и промежуточные суммы.

In [ ]:
import numpy as np
def show_metrics(m, name):
    y = m.y.values
    y_pred = m.predict()
    n, k = len(y), m.x.shape[1]
    e = y - y_pred
    ss_res = np.sum(e**2)
    ss_tot = np.sum((y - y.mean())**2)
    
    mse = ss_res / n
    rmse = np.sqrt(mse)
    mae = np.sum(np.abs(e)) / n
    mape = np.sum(np.abs(e / y)) / n * 100
    rmsle = np.sqrt(np.sum((np.log(1 + y) - np.log(1 + np.maximum(y_pred, 0)))**2) / n)
    r2 = 1 - ss_res / ss_tot
    r2_adj = 1 - (ss_res / (n - k - 1)) / (ss_tot / (n - 1))
    aic = n * np.log(mse) + 2 * (k + 1)
    bic = n * np.log(mse) + (k + 1) * np.log(n)
    
    print(name)
    print("MSE:    ", round(mse, 2))
    print("RMSE:   ", round(rmse, 2))
    print("MAE:    ", round(mae, 2))
    print("MAPE:   ", round(mape, 2), "%")
    print("RMSLE:  ", round(rmsle, 4))
    print("R^2:    ", round(r2, 5))
    print("R^2 adj:", round(r2_adj, 5))
    print("AIC:    ", round(aic, 2))
    print("BIC:    ", round(bic, 2))
    print()

show_metrics(m1, "--- Метрики до очистки ---")
show_metrics(m2, "--- Метрики после очистки ---")

--- Метрики до очистки ---
MSE:     169208461720.94
RMSE:    411349.56
MAE:     89934.75
MAPE:    643.8 %
RMSLE:   1.8375
R^2:     0.0054
R^2 adj: 7e-05
AIC:     9727.25
BIC:     9739.04

--- Метрики после очистки ---
MSE:     673095203.45
RMSE:    25944.08
MAE:     20604.45
MAPE:    234.68 %
RMSLE:   1.2272
R^2:     0.10438
R^2 adj: 0.09895
AIC:     6775.02
BIC:     6786.45

**Вывод по метрикам:**
* Отрицательных прогнозов $\hat{y}$ нет, поэтому стандартная RMSLE применима.
* Квадратичные метрики (MSE, RMSE) наиболее чувствительны к выбросам и уменьшились в 16-250 раз.
* Информационные критерии AIC и BIC снизились почти на 3000 пунктов, подтверждая преимущество очищенной модели.

### Итоговый вывод
1. **Значимость:** До очистки модель была незначима ($F = 1.01, p = 0.36$). Очистка от выбросов сделала модель статистически значимой ($F = 19.23, p < 0.001$).
2. **Факторы:** И доход ($p < 0.001$), и число детей ($p = 0.0037$) значимо увеличивают объем займов и расходуемых накоплений.
3. **Экономический смысл:** Состоятельные домохозяйства привлекают более крупные займы (на активы/покупки), а семьи с детьми активнее используют заемные средства для компенсации возросших текущих расходов.

### Задание 6* (Дополнительно). Степенная модель в логарифмах
Оценим модель $\ln(y) = \gamma_0 + \gamma_1 \ln(x_1) + \gamma_2 \ln(x_2 + 1) + u$ через наш класс `Model`:

In [ ]:
df_clean_log = df_clean.copy()
df_clean_log['rsber'] = np.log(df_clean_log['rsber'])
df_clean_log['doxodn'] = np.log(df_clean_log['doxodn'])
df_clean_log['chdet'] = np.log(df_clean_log['chdet'] + 1)

# Оцениваем степенную модель через наш класс Model
m_log = Model(df_clean_log)
m_log.fit()
m_log.f_fact()
m_log.t_fact()

print("Коэффициенты (эластичности):")
names_log = ['const', 'ln(doxodn)', 'ln(chdet+1)']
for i in range(3):
    print(names_log[i] + ":", "gamma =", round(m_log.coefs[i], 4), "SE =", round(m_log.se[i], 4), "t =", round(m_log.t_stats[i], 3), "p-value =", round(m_log.t_p_values[i], 4))

print()
print("R^2:         ", round(m_log.r2, 4))
print("F-статистика:", round(m_log.f, 2))
print("p-значение:  ", round(m_log.p_value, 8))
m_log.check()

NameError: name 'df_clean' is not defined

**Интерпретация эластичностей:**
* Эластичность по доходу $\gamma_1 = 1.0748$ ($t = 5.863, p < 0.001$): при росте дохода на 1% сумма займов и сбережений растет в среднем на 1.07%.
* Эластичность по детям $\gamma_2 = 0.5581$ ($t = 3.108, p = 0.0020$): при росте показателя детей на 1% привлекаемые средства увеличиваются на 0.56%.
* Обе эластичности статистически значимы по критериям Стьюдента.